In [52]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from catboost import CatBoostRegressor

test = pd.read_csv('Full_cleaned_test.csv')
train = pd.read_csv('Full_cleaned_train.csv')

In [53]:
# Feature Engineering
for df in [train, test]:
    df['power_output'] = df['voltage'] * df['current']
    df['temp_diff'] = df['module_temperature'] - df['temperature']
    df['irradiance_per_cloud'] = df['irradiance'] / (df['cloud_coverage'] + 1)
    
train['power_output'] = train['voltage'] * train['current']
train['temp_diff'] = train['module_temperature'] - train['temperature']
train['irradiance_per_cloud'] = train['irradiance'] / (train['cloud_coverage'] + 1)

    
test['power_output'] = test['voltage'] * test['current']
test['temp_diff'] = test['module_temperature'] - test['temperature']
test['irradiance_per_cloud'] = test['irradiance'] / (test['cloud_coverage'] + 1)

In [54]:
train.shape

(19369, 20)

In [58]:

# Feature selection
drop_cols = ['id', 'efficiency','voltage']
X = train.drop(columns=drop_cols)
y = train['efficiency']
X_test = test.drop(columns=['id','voltage'])

# Train/validation split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)


In [59]:


# 1. Best params: {'depth': 4, 'iterations': 500, 'learning_rate': 0.03}
model = CatBoostRegressor(iterations=300, learning_rate=0.07, depth=5, random_seed=42, verbose=False)
model.fit(X_train, y_train, eval_set=(X_val, y_val), early_stopping_rounds=50)

# Validation score
y_pred = model.predict(X_val)
rmse = np.sqrt(mean_squared_error(y_val, y_pred))
score = (1 - rmse) * 100
print(f"RMSE: {rmse:.4f}, Score: {score:.8f}")

RMSE: 0.0464, Score: 95.35724803


In [60]:

from datetime import datetime
# Get current date and time
now = datetime.now()
timestamp = now.strftime("%H%M%S")
print(f"submission_{timestamp}")

# Final prediction
preds = model.predict(X_test)
submission = pd.DataFrame({'id': test['id'], 'efficiency': preds})

submission.to_csv(f'submission_{timestamp}.csv', index=False)
submission.shape

submission_163208


(12000, 2)

In [47]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.layers import BatchNormalization


# Scale numeric features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_test_scaled = scaler.fit_transform(X_test)

# model = Sequential([
#     Dense(128, activation='relu', input_shape=(X_scaled.shape[1],)),
#     Dropout(0.2),
#     Dense(64, activation='relu'),
#     Dense(1)  # No activation for regression
# ])



model = Sequential([
    Dense(256, activation='relu', input_shape=(X_scaled.shape[1],)),
    BatchNormalization(),
    Dropout(0.3),
    Dense(128, activation='relu'),
    BatchNormalization(),
    Dropout(0.2),
    Dense(64, activation='relu'),
    Dense(1)
])

model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.fit(X_scaled, y, epochs=200, batch_size=64, validation_split=0.2)


Epoch 1/200


c:\Users\vidha\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


243/243 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - loss: 0.1337 - mae: 0.2221 - val_loss: 0.0134 - val_mae: 0.0940
Epoch 2/200
243/243 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 0.0129 - mae: 0.0894 - val_loss: 0.0120 - val_mae: 0.0879
Epoch 3/200
243/243 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 0.0125 - mae: 0.0879 - val_loss: 0.0116 - val_mae: 0.0856
Epoch 4/200
243/243 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 0.0121 - mae: 0.0865 - val_loss: 0.0118 - val_mae: 0.0852
Epoch 5/200
243/243 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 0.0122 - mae: 0.0868 - val_loss: 0.0129 - val_mae: 0.0883
Epoch 6/200
243/243 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 0.0119 - mae: 0.0859 - val_loss: 0.0118 - val_mae: 0.0866
Epoch 7/200
243/243 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 0.0117 - mae: 0.0853 - val_loss: 0.0116 - val_mae: 0.0857
Epoch 8/200
243/243 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 0.0120 - mae: 0.0865 - val_loss: 0.0116 - val_mae: 0.0852
Epoch 9/200
243/243 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss:

In [34]:
X_test_scaled = scaler.fit_transform(X_test)
X_test_scaled

array([[-0.66492721, -1.69412356, -0.30538561, ...,  0.44517676,
        -0.41829448,  1.33895113],
       [ 0.80050205,  0.89915405, -1.69524765, ...,  1.33776173,
        -1.24975611, -1.34685935],
       [ 0.711136  , -0.06641936,  1.43779587, ...,  1.33776173,
        -0.41829448, -0.45158919],
       ...,
       [-0.11675596,  1.04083031, -0.93434141, ..., -1.33999319,
        -1.24975611, -1.34685935],
       [-0.83479207, -1.32009044, -1.71558393, ..., -0.44740822,
         1.24462876, -0.45158919],
       [-0.99143236,  0.75605998,  0.03741772, ..., -1.33999319,
         1.24462876, -0.45158919]], shape=(12000, 15))

In [48]:
# Train/validation split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
# Validation score
y_pred = model.predict(X_val)
rmse = np.sqrt(mean_squared_error(y_val, y_pred))
score = (1 - rmse) * 100
print(f"RMSE: {rmse:.4f}, Score: {score:.8f}")

122/122 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
RMSE: 0.1068, Score: 89.31863063


In [49]:
X_val

,temperature,irradiance,humidity,panel_age,maintenance_count,soiling_ratio,voltage,current,module_temperature,cloud_coverage,wind_speed,pressure,string_id,error_code,installation_type
10481,22.783119,538.761564,55.396841,33.262714,3.6,0.596869,33.334579,1.026133,28.725907,63.640102,11.744263,996.295837,0,0,2
3189,18.709085,242.873940,26.908976,4.114522,4.0,0.997875,17.471391,1.448668,25.193990,18.391466,12.609947,996.181378,2,3,1
15362,31.557522,657.932519,17.358039,26.174060,7.0,0.494802,3.252189,1.008342,42.101677,54.068220,3.537823,1005.101101,2,0,2
9786,48.694784,374.195734,3.512369,7.107351,1.0,0.578578,21.752837,0.529268,46.887748,54.788292,7.254400,1015.312246,0,1,3
4113,33.488459,625.203283,66.421386,15.217988,3.0,0.912205,51.256035,1.889493,40.823465,26.567744,4.044422,1008.567676,1,2,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14123,41.375605,183.826925,9.102022,20.485493,3.8,0.672984,15.207089,1.499592,39.854059,54.968383,5.262446,995.622845,3,3,0
16960,22.701824,507.764373,15.389479,23.592378,5.0,0.861768,7.094697,2.427238,31.085755,23.821898,10.196035,1009.723206,0,1,0
8499,29.245176,270.806031,44.503251,7.637097,8.0,0.568537,22.610444,2.072481,36.475095,91.106704,0.949539,1028.108337,3,1,2
8486,27.827452,-34.751181,77.239330,0.118556,4.0,0.491954,23.441974,0.388043,39.843201,18.218508,7.491279,1013.783954,2,0,1


In [50]:

from datetime import datetime
# Get current date and time
now = datetime.now()
timestamp = now.strftime("%H%M%S")
print(f"submission_{timestamp}")

# Scale numeric features
# scaler = StandardScaler()
# X_scaled_test = scaler.fit_transform(X_test)

# Final prediction
# preds = model.predict(X_val)



submission_162807


In [51]:
y_pred = model.predict(X_test_scaled)
y_pred = pd.DataFrame(y_pred)
submission = pd.DataFrame({'id': test['id'], 'efficiency': y_pred[0]})
submission.to_csv(f'dl_submission_{timestamp}.csv', index=False)
submission.shape

375/375 ━━━━━━━━━━━━━━━━━━━━ 0s 775us/step


(12000, 2)